# Text-based Machine Learning

---

> Machine Learning is “A field of computer science that gives computers the ability to learn from data without being explicitly programmed.” — Wikipedia

### Types of ML:
| Type                  | Description                                                | Examples                          |
|-----------------------|------------------------------------------------------------|-----------------------------------|
| **Supervised**        | Data has known labels; learn input-output mapping          | Classification, Regression        |
| **Unsupervised**      | No labels; discover structure from data                    | Clustering, Topic Modeling        |
| **Reinforcement**     | Learn by interacting with environment to maximize reward   | Game AI, Robotics                 |

---

# Feature Engineering Overview
> Feature enginnering: the process of converting text into vectors (numbers)

> Foundation: Vector Space Model (a.k.a Term Vector Model), vsm for short, is a mathmatical model for representing text documents as vectors in a multi-dimensional space.这是后续几个模型的理论基础，把文档表示为为不同维的向量，每一个维度表示一个词，**值**表示这个词在文档中的权重（例如：词频frequency, 出现次数count, TF-IDF, 是否出现binary(0/1)）
- Documents are represented as vectors:
  `D_j = [w1j, w2j, ..., wnj]`
- Each dimension is a **term (word)**; each value is the **weight** of that term in the document (e.g., frequency, TF-IDF)

---

### 1. Bag-of-Words (BoW) Model - unigram

- Ignores grammar, order, punctuation — treats document as a “bag” of words
- Creates a Document-Term Matrix (DTM)
- Can be built with:
  - **Keras**: `texts_to_matrix()` with modes like `binary`, `count`, `freq`, `tfidf`
  - **Scikit-learn**: `CountVectorizer()` ➝ sparse matrix

    ### 🔢 Keras Example
    ```python
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(sentences)
    word_bag = tokenizer.texts_to_matrix(sentences, mode='binary')

### 2. N-Gram Model
- Extension of BoW to sequences of n words
- Common n-grams: bigrams (2-grams), trigrams (3-grams)
- Implemented with CountVectorizer(ngram_range=(n, n))



Example:

If corpus is: `["I love cats", "I hate cats"]` \
vocabulary = `{'I': 1, 'love': 2, 'hate': 3, 'cats': 4}` \
Then the document-term matrix is (unigram / bi-gram / trigram / N-gram):
```
|      | I | love | hate | cats |
|------|---|------|------|------|
| 1    | 1 | 1    | 0    | 1    |
| 2    | 1 | 0    | 1    | 1    |
```
And `I love cats` becomes `[1, 1, 0, 1]` and `I hate cats` becomes `[1, 0, 1, 1]`

# Different modes
## TF-IDF (Term Frequency-Inverse Document Frequency)
- Measures importance of a term in a document relative to a corpus
- TF-IDF = TF * IDF
> TF = term frequency (how often a term appears in a document)
> IDF = inverse document frequency (how common or rare a term is across all documents)
> 即「词在当前文档中出现得多 × 在所有文档中不常见」

### Tfidf的三种实现方式
- From **keras** Tokenizer: It is by default unigram
    - ```python
      from keras.preprocessing.text import Tokenizer
      word_bag = tokenizer.texts_to_matrix(sentence_list, mode='tfidf')
      ```

- From **sklearn** CountVectorizer
    - ```python
      from sklearn.feature_extraction.text import CountVectorizer
      cv = CountVectorizer(min_df=0., max_df=1.) # min/max_df: min/max frequency of a token
      # 可以用min_df=0.1, max_df=0.9来过滤掉一些高频词和低频词
      ```
- From **sklearn** TfidfVectorizer
    - ```python
      from sklearn.feature_extraction.text import TfidfVectorizer
      vectorizer = TfidfVectorizer()
      unigram_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))
      bigram_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
      # n-gram_range=(1, 2) means unigram + bigram
      # (1,1) means unigram only
      # (2,2) means bigram only
      # (1,3) means unigram + bigram + trigram
      ```


# Different representation - in BoW
### sparse matrix 稀疏矩阵
每一行表示一个 (行号, 列号) 值，也就是：\
所有非0（row, column) value）\
“第几篇文档的第几个词出现了几次”
### dense matrix 密集矩阵
每一行就是一个文档的词向量（Document-Term Vector）

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = ["I love NLP NLP NLP", "NLP is fun", "I love machine learning"]

# 初始化 CountVectorizer（使用 unigram）
vectorizer = CountVectorizer()

# 拟合并转换为稀疏矩阵
sparse_matrix = vectorizer.fit_transform(corpus)

# 查看词汇表（列名）
print("Vocabulary (Features):")
print(vectorizer.get_feature_names_out())

# 打印稀疏矩阵
print("\nSparse Matrix (compressed format):")
print(sparse_matrix)

"""
  (0, 3)  1
  (0, 5)  1
  (1, 0)  1
  (1, 1)  1
  (1, 5)  1
  (2, 2)  1
  (2, 3)  1
  (2, 4)  1
  (2, 5)  1
"""

# 转换成密集矩阵
dense_matrix = sparse_matrix.toarray()

"""
[
  [0, 0, 0, 1, 0, 1],  # "I love NLP"
  [1, 1, 0, 0, 0, 1],  # "NLP is fun"
  [0, 0, 1, 1, 1, 1]   # "I love machine learning"
]
"""
# 打印密集矩阵
print("\nDense Matrix (normal array):")
print(dense_matrix)


Vocabulary (Features):
['fun' 'is' 'learning' 'love' 'machine' 'nlp']

Sparse Matrix (compressed format):
  (0, 3)	1
  (0, 5)	3
  (1, 5)	1
  (1, 1)	1
  (1, 0)	1
  (2, 3)	1
  (2, 4)	1
  (2, 2)	1

Dense Matrix (normal array):
[[0 0 0 1 0 3]
 [1 1 0 0 0 1]
 [0 0 1 1 1 0]]





## Word2Vec
- Word2Vec is a technique to represent words as vectors in a continuous vector space
- It captures semantic meaning and relationships between words
- Trained on large corpus of text
- Two main architectures:
  - Continuous Bag of Words (CBOW): Predicts target word from context words
  - Skip-gram: Predicts context words from target word

##